# TTS 캐시 사전 생성 (Colab)

`static/tts_cache/food_{food_id}_{STYLE}.wav` 50개를 다시 만든다. 내레이션 문구가 바뀌면
(`PRONUNCIATION_TTS` 도입, `LORA_SCALE` 변경 등) 기존 wav가 낡으므로 전량 재생성해야 한다 —
`src/pipeline.py`의 `create_tts_response()`가 캐시를 먼저 보기 때문에, 파일을 안 바꾸면
코드를 고쳐도 사용자에게는 아무것도 안 바뀐다.

**왜 `python -m src.tts.pregenerate`를 그대로 안 쓰나**: 그 모듈은 `src/pipeline.py`를
임포트하고, 그게 `src/ocr/infer.py` -> `paddle`/`paddleocr`을 모듈 수준에서 끌어온다.
Colab에 paddle을 까는 건 무겁고, paddle과 torch를 한 프로세스에 올리는 건 이 프로젝트가
별도 워커 프로세스로 피해온 바로 그 충돌이다(`src/tts/worker.py` 참고).

대신 워커가 실제로 호출하는 `src.tts.infer.synthesize()`를 직접 부른다. 워커는 그 함수를
JSON 한 줄로 감싼 것뿐이라, 나오는 음성은 라이브 경로와 같다.

**런타임**: 상단 메뉴 > 런타임 > 런타임 유형 변경 > GPU (T4면 충분).
**소요 시간**: 설치·모델 로딩 약 10분 + 생성 50개 약 30분.

## 0. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. 패키지 설치

`requirements.txt`의 TTS 관련 항목만 깐다 — paddle·opencv 등 OCR 쪽은 필요 없다.

In [ ]:
!pip install -q -U qwen-tts peft torchao huggingface_hub soundfile safetensors supabase python-dotenv

## 2. 저장소 가져오기

저장소가 비공개라 그냥 clone은 안 된다. 아래 **A(Drive)** 와 **B(GitHub 토큰)** 중 하나만
실행하면 된다. 토큰을 새로 만들 필요가 없는 A를 권한다.

### 2-A. Google Drive에서 가져오기 (권장)

내 PC에서 저장소 폴더를 통째로 압축해(`proj1-c.zip`) Drive의
`MyDrive/k-menu-voice-guide/`에 올려둔다. 팀이 이미 쓰는 폴더다.

`.venv`·`models`·`data/static` 같은 큰 폴더는 빼도 된다 — 이 노트북은 `src/` 아래 파이썬
파일만 쓴다. 용량을 줄이려면 `src` 폴더만 압축해도 되고, 그 경우 아래 `ZIP_PATH`가
가리키는 zip 안에 `src/`가 최상위로 들어 있으면 된다.

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/k-menu-voice-guide/proj1-c.zip'  # 실제 경로로 수정
REPO_DIR = '/content/proj1-c'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)   # 항상 최신 zip 기준으로 다시 푼다
Path(REPO_DIR).mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(REPO_DIR)

# zip 안에 폴더가 한 겹 더 있으면(proj1-c/src/...) 그 안쪽을 쓴다
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    inner = [p for p in Path(REPO_DIR).iterdir() if p.is_dir()]
    if len(inner) == 1 and (inner[0] / 'src').is_dir():
        REPO_DIR = str(inner[0])

os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)
assert os.path.isfile('src/tts/infer.py'), 'src/tts/infer.py가 없다 - zip 내용을 확인할 것'
print('src/tts/infer.py 확인 완료')

### 2-B. GitHub에서 clone (A를 썼으면 건너뛴다)

토큰이 필요하다. GitHub > 우측 상단 프로필 > **Settings** > 맨 아래 **Developer settings** >
**Personal access tokens** > **Fine-grained tokens** > *Generate new token* 에서

- Repository access: **Only select repositories** > `aihuman-7th/proj1-c`
- Permissions > Repository permissions > **Contents: Read-only**
- Expiration: 짧게 (이 작업만 하고 버릴 토큰이다)

로 만든 뒤, Colab 왼쪽 **열쇠 아이콘(비밀)** 에 `GH_TOKEN`으로 넣고 노트북 접근을 켠다.
노트북에 직접 적지 않는다 — 이 노트북은 저장소에 커밋되기 때문이다.

In [ ]:
import os, subprocess
from google.colab import userdata

BRANCH = 'feat/tts-hangul-labeling'
REPO_DIR = '/content/proj1-c'

if not os.path.exists(REPO_DIR):
    token = userdata.get('GH_TOKEN')
    url = f'https://{token}@github.com/aihuman-7th/proj1-c.git'
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', url, REPO_DIR], check=True)
    print('clone 완료')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('이미 있음 - 최신으로 갱신')

os.chdir(REPO_DIR)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

## 3. Supabase 접속 정보

역시 Colab 비밀에 `SUPABASE_URL`, `SUPABASE_ANON_KEY`를 넣는다. 읽기 전용 anon key다.
`src/db/supabase_client.py`가 `os.environ`에서 읽으므로 환경변수로만 올려주면 된다.

In [ ]:
from google.colab import userdata

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_ANON_KEY"] = userdata.get("SUPABASE_ANON_KEY")

import sys
sys.path.insert(0, REPO_DIR)

from src.db import cache
print("foods:", len(cache.get_foods()))
cats = {}
for r in cache._load()["food_details"]:
    cats[r["category"]] = cats.get(r["category"], 0) + 1
print("food_details 카테고리:", cats)
assert cats.get("PRONUNCIATION_TTS", 0) > 0, (
    "PRONUNCIATION_TTS 행이 없다 - food_details에 그 카테고리를 먼저 넣을 것. "
    "이대로 생성하면 화면용 발음 문장이 그대로 들어가 이름을 두 번 읽는다."
)

## 4. 생성할 목록 확인

`get_narration()`이 실패하는 음식(내레이션·상세 데이터가 아직 없는 신규 음식)은 건너뛴다.
생성 전에 텍스트를 눈으로 한 번 훑는다 — 여기서 이상하면 wav 50개가 다 이상해진다.

In [ ]:
from src.text_gen.narration import get_narration

targets, skipped = [], []
for food in sorted(cache.get_foods(), key=lambda f: f["id"]):
    try:
        n = get_narration(food["id"])
    except ValueError as exc:
        skipped.append((food["id"], food["name_kr"], str(exc)))
        continue
    targets.append((food["id"], food["name_kr"], n["style"], n["tts_text"]))

print(f"생성 대상 {len(targets)}개 / 내레이션 없어 건너뜀 {len(skipped)}개\n")
for fid, kr, style, text in targets[:3]:
    print(f"[{fid}] {kr} ({style})\n  {text}\n")

## 5. 50개 생성

첫 요청에 모델을 GPU에 올리느라 90~160초가 더 걸리고, 그 뒤로는 개당 30초 안팎이다.
이미 만든 파일은 건너뛰므로, 런타임이 끊겨도 이 셀만 다시 돌리면 이어서 진행된다.
전부 다시 만들려면 `FORCE = True`로 둔다.

In [ ]:
import shutil, time
from pathlib import Path
from src.tts import infer

FORCE = True   # 문구·LORA_SCALE이 바뀐 뒤라면 True. 중단 후 이어하기라면 False.

OUT_DIR = Path(REPO_DIR) / "static" / "tts_cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"LORA_SCALE={infer.LORA_SCALE} / speaker={infer.TTS_SPEAKER_NAME}")
print(f"출력: {OUT_DIR}\n")

done = skip = fail = 0
for i, (fid, kr, style, text) in enumerate(targets, 1):
    target = OUT_DIR / f"food_{fid}_{style}.wav"
    label = f"[{i}/{len(targets)}] {kr} ({style})"
    if target.exists() and not FORCE:
        print(f"{label} - 있음, 건너뜀"); skip += 1; continue
    try:
        started = time.perf_counter()
        result = infer.synthesize(text, style)
        shutil.move(result["audio_path"], target)
        print(f"{label} - 생성 {time.perf_counter() - started:.1f}초")
        done += 1
    except Exception as exc:
        print(f"{label} - 실패: {type(exc).__name__}: {exc}")
        fail += 1

print(f"\n생성 {done} / 건너뜀 {skip} / 실패 {fail}")

## 6. 확인 후 내려받기

파일 개수와 길이를 확인하고, 몇 개는 직접 들어본다 — 음식명이 한국어로 들리는지,
같은 이름을 두 번 읽지 않는지가 이번 변경의 핵심이다.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

wavs = sorted(OUT_DIR.glob("*.wav"), key=lambda p: int(p.stem.split("_")[1]))
print(f"wav {len(wavs)}개")
total = 0.0
for p in wavs:
    info = sf.info(str(p))
    total += info.frames / info.samplerate
print(f"총 길이 {total/60:.1f}분 / 평균 {total/len(wavs):.1f}초\n")

# CAUTION 1개 + GUIDE 2개를 들어본다
sample_ids = [3, 1, 47]
for fid, kr, style, text in targets:
    if fid in sample_ids:
        print(f"[{fid}] {kr} ({style})\n  {text}\n")
        display(Audio(str(OUT_DIR / f"food_{fid}_{style}.wav")))

In [ ]:
from google.colab import files

shutil.make_archive("/content/tts_cache", "zip", OUT_DIR)
print("zip 생성:", Path("/content/tts_cache.zip").stat().st_size / 1e6, "MB")
files.download("/content/tts_cache.zip")

## 7. 반영

내려받은 `tts_cache.zip`을 풀어 저장소의 `static/tts_cache/`에 덮어쓴다.
`src/pipeline.py`의 `_load_cached_tts()`가 `food_{food_id}_{STYLE}.wav` 이름으로 찾으므로
파일명은 그대로 두어야 한다.

배포 Space에도 같은 파일이 올라가야 사용자에게 반영된다.